# # Chicago Taxi Trips — week of 11–17 September 2023
#
# **Summary** — _to be filled in once the analysis is complete._

# ## 1. Data loading and cleaning
#
# All figures in this notebook describe **reported trips** in `trips.csv` (Mon 2023-09-11 to Sun 2023-09-17), not total taxi market volume — the City notes that not every trip is reported.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# One restrained house style: a single blue for magnitude, recessive grid, no box.
BLUE, ORANGE, GREY, INK = "#2a78d6", "#eb6834", "#9a9992", "#2b2b2a"
plt.rcParams.update({
    "figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.edgecolor": GREY, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": INK, "ytick.color": INK, "axes.titleweight": "bold",
    "axes.titlesize": 12, "axes.titlelocation": "left", "legend.frameon": False,
})
FIG = Path("figures/analysis")
FIG.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv("trips.csv", parse_dates=["trip_start_timestamp", "trip_end_timestamp"])
print("shape:", raw.shape)
print("trip_id unique:", raw.trip_id.is_unique, "| fully duplicated rows:", raw.duplicated().sum())
print("start:", raw.trip_start_timestamp.min(), "->", raw.trip_start_timestamp.max())

shape: (135768, 23)
trip_id unique: True | fully duplicated rows: 0
start: 2023-09-11 00:00:00 -> 2023-09-17 23:45:00


In [2]:
miss = raw.isna().sum()
print("Columns with missing values (n, % of 135,768 rows):")
print(pd.DataFrame({"n_missing": miss, "pct": miss / len(raw) * 100})[miss > 0])

Columns with missing values (n, % of 135,768 rows):
                            n_missing   pct
trip_end_timestamp                  1  0.00
trip_seconds                       33  0.02
pickup_census_tract             71404 52.59
dropoff_census_tract            73842 54.39
pickup_community_area            3815  2.81
dropoff_community_area          13794 10.16
fare                              324  0.24
tips                              324  0.24
tolls                             324  0.24
extras                            324  0.24
trip_total                        324  0.24
pickup_centroid_latitude         3775  2.78
pickup_centroid_longitude        3775  2.78
pickup_centroid_location         3775  2.78
dropoff_centroid_latitude       12872  9.48
dropoff_centroid_longitude      12872  9.48
dropoff_centroid_location       12872  9.48


In [3]:
r = raw
mph = r.trip_miles / (r.trip_seconds / 3600)
checks = {
    "trip_seconds == 0": (r.trip_seconds == 0).sum(),
    "trip_seconds missing": r.trip_seconds.isna().sum(),
    "trip_seconds > 6 hours": (r.trip_seconds > 6 * 3600).sum(),
    "trip_miles == 0": (r.trip_miles == 0).sum(),
    "trip_miles == 0 but fare > $10": ((r.trip_miles == 0) & (r.fare > 10)).sum(),
    "trip_miles > 100": (r.trip_miles > 100).sum(),
    "implied speed > 80 mph (miles & seconds > 0)": (mph > 80).sum(),
    "fare/tips/tolls/extras/trip_total all missing": r.trip_total.isna().sum(),
    "trip_total == 0": (r.trip_total == 0).sum(),
    "fare > $200": (r.fare > 200).sum(),
    "trip_total > $500": (r.trip_total > 500).sum(),
    "tolls > $100": (r.tolls > 100).sum(),
    "extras > $100": (r.extras > 100).sum(),
}
print(pd.Series(checks, name="rows").to_frame().assign(pct=lambda d: d.rows / len(r) * 100).to_string(float_format=lambda x: f"{x:,.2f}"))
print()
print("Most extreme money rows (by trip_total):")
print(r.sort_values("trip_total", ascending=False)[["trip_seconds","trip_miles","fare","tips","tolls","extras","trip_total","payment_type"]].head(6))

                                                rows   pct
trip_seconds == 0                               2731  2.01
trip_seconds missing                              33  0.02
trip_seconds > 6 hours                            98  0.07
trip_miles == 0                                13993 10.31
trip_miles == 0 but fare > $10                  5778  4.26
trip_miles > 100                                  11  0.01
implied speed > 80 mph (miles & seconds > 0)     228  0.17
fare/tips/tolls/extras/trip_total all missing    324  0.24
trip_total == 0                                  134  0.10
fare > $200                                       49  0.04
trip_total > $500                                  8  0.01
tolls > $100                                       3  0.00
extras > $100                                     15  0.01

Most extreme money rows (by trip_total):
        trip_seconds  trip_miles   fare  tips    tolls   extras  trip_total payment_type
124235      1,080.00        0.10  11.25  0.

In [4]:
parts = r[["fare", "tips", "tolls", "extras"]].sum(axis=1, min_count=4)
gap = (r.trip_total - parts).round(2)
print("trip_total minus (fare + tips + tolls + extras), rows with all money fields present:", gap.notna().sum())
print(gap.value_counts().head(8).rename("rows").to_frame().T.to_string())
print()
print("Gap < 0:", (gap < -0.005).sum(), "| gap > $3:", (gap > 3.005).sum(), "| gap > $5:", (gap > 5).sum())
print("Rows with a non-zero gap, by payment type (share of that type):")
print((gap.abs() > 0.005).groupby(r.payment_type).mean().round(3).to_dict())
print(f"Total gap ${gap.sum():,.0f} = {gap.sum() / r.trip_total.sum():.2%} of reported trip_total")

trip_total minus (fare + tips + tolls + extras), rows with all money fields present: 135444
       0.00   0.50  3.00  2.50  1.00  1.95  1.50  2.00
rows  87247  46522   804   751   109     5     4     2

Gap < 0: 0 | gap > $3: 0 | gap > $5: 0
Rows with a non-zero gap, by payment type (share of that type):


{'Cash': 0.017, 'Credit Card': 0.719, 'Dispute': 0.0, 'Mobile': 0.224, 'No Charge': 0.0, 'Prcard': 0.001, 'Unknown': 0.0}
Total gap $27,679 = 0.68% of reported trip_total


In [5]:
vc = r.company.value_counts()
print("Distinct company labels:", len(vc))
print("Labels that look like spelling variants of one another, with trips and distinct taxis:")
variants = ["Taxicab Insurance Agency Llc", "Taxicab Insurance Agency, LLC", "Choice Taxi Association", "Choice Taxi Association Inc",
            "Taxi Affiliation Services", "Taxi Affiliation Services Llc - Yell"]
tx = r.groupby("company").taxi_id.agg(trips="size", taxis="nunique").loc[variants]
print(tx)
sets = r.groupby("company").taxi_id.agg(set)
for a, b in [variants[0:2], variants[2:4], variants[4:6]]:
    print(f"taxi_ids shared by '{a}' and '{b}':", len(sets[a] & sets[b]))
print("taxis reporting under more than one company label:", (r.groupby("taxi_id").company.nunique() > 1).sum(), "of", r.taxi_id.nunique())

Distinct company labels: 32
Labels that look like spelling variants of one another, with trips and distinct taxis:
                                      trips  taxis
company                                           
Taxicab Insurance Agency Llc          13914    294
Taxicab Insurance Agency, LLC          2379     62
Choice Taxi Association                2022     50
Choice Taxi Association Inc            1064     20
Taxi Affiliation Services             25727    457
Taxi Affiliation Services Llc - Yell     75      2
taxi_ids shared by 'Taxicab Insurance Agency Llc' and 'Taxicab Insurance Agency, LLC': 0
taxi_ids shared by 'Choice Taxi Association' and 'Choice Taxi Association Inc': 0
taxi_ids shared by 'Taxi Affiliation Services' and 'Taxi Affiliation Services Llc - Yell': 0


taxis reporting under more than one company label: 3 of 2656


# ### Cleaning decisions and assumptions
#
# Two bases are used, so every number in the notebook is on one of them:
#
# * **Trip basis — all 135,768 reported trips.** Used for every *count* (demand by day/hour, pickup areas, trips per payment type, trips per company). No duplicates (`trip_id` is unique), and every trip starts inside the Mon–Sun week, so nothing is dropped here.
# * **Priced basis — trips with a usable `trip_total`.** Used for every *dollar* figure (revenue, average fare, tips). Excluded from this basis, in this order:
#   1. **324 trips with no fare / tips / tolls / extras / trip_total at all** (323 of them cash) — real trips, but unpriced.
#   2. **134 trips with `trip_total` = 0** — no-charge, disputed or failed-payment trips; they add no revenue and would drag down averages.
#   3. **8 trips with `trip_total` > $500** — includes tolls of \$4,444.44 / \$4,111.10 on trips of a mile or so (clearly keyed errors worth ~\$21,900, or 0.5% of all revenue, which would distort company totals) and a \$758–\$911 fare on a few-second or 37-mile trip. The next-highest total is \$483 for a 195-mile trip, which is plausible.
#
#    Because these rows still count as trips, **"revenue" always means revenue on priced trips**, and tables show how many trips that is.
#
# **Looked at but deliberately *not* excluded**
# * **Miles and duration are not used in any question**, so zero/odd values do not affect results: 13,993 trips with 0 miles (5,778 of them with a fare over \$10), 2,731 with 0 seconds, 98 over 6 hours, 11 over 100 miles. Fares for these look like real metered or keyed flat fares; I treat `fare`/`trip_total` as authoritative.
# * **Large fares on near-zero-distance card trips** (e.g. \$195–\$400 with 0 miles) look like manually keyed flat fares (the tips are a plausible 20% of them). Kept; they are a small share of revenue.
# * **13 trips with `extras` over \$100** (all with `trip_total` ≤ \$383) — odd but small (≈0.04% of revenue). Kept.
# * **`trip_total` vs parts**: `trip_total` is never below fare + tips + tolls + extras and never more than \$3 above it (no row differs by more than \$5). 46,522 rows differ by exactly \$0.50, mostly card/mobile trips — presumably a small surcharge that isn't itemised. I treat `trip_total` as the reported trip revenue (0.68% higher than the sum of its parts overall) and do not adjust it.
#
# **Other assumptions**
# * **Timestamps are Chicago local time** (as published by the City) and are rounded to the nearest 15 minutes, so a trip's "hour" is only accurate to about ±7 minutes. Day and hour come from `trip_start_timestamp`.
# * **Company names:** two pairs are the same company spelled two ways — "Taxicab Insurance Agency Llc" / "…, LLC" and "Choice Taxi Association" / "… Inc". I merge them (they share no `taxi_id`, so summing taxi counts is exact). "Taxi Affiliation Services Llc - Yell" (75 trips, 2 taxis) may be Yellow Cab's branch of "Taxi Affiliation Services", but I can't confirm that, so it stays separate; it doesn't affect the top 10.
# * **Geography uses `pickup_community_area` only** (census tracts are blank for 53% of trips). Trips with no pickup community area are kept as their own "none" group rather than dropped.
# * `Prcard` is assumed to mean a prepaid card. `Unknown`, `No Charge` and `Dispute` are kept as reported.

In [6]:
df = raw.copy()

# company spelling variants
company_map = {"Taxicab Insurance Agency, LLC": "Taxicab Insurance Agency Llc",
               "Choice Taxi Association Inc": "Choice Taxi Association"}
df["company_clean"] = df.company.replace(company_map)

# priced basis, as a waterfall
n_total = len(df)
missing_money = df.trip_total.isna()
zero_total = ~missing_money & (df.trip_total == 0)
huge_total = ~missing_money & (df.trip_total > 500)
df["priced"] = ~(missing_money | zero_total | huge_total)

# time fields
df["date"] = df.trip_start_timestamp.dt.normalize()
df["dow"] = df.trip_start_timestamp.dt.day_name()
df["hour"] = df.trip_start_timestamp.dt.hour

print(f"Trip basis (all trips):               {n_total:>8,}")
print(f"  excluded: missing money fields       {missing_money.sum():>8,}")
print(f"  excluded: trip_total == 0            {zero_total.sum():>8,}")
print(f"  excluded: trip_total > $500          {huge_total.sum():>8,}")
print(f"Priced basis:                         {df.priced.sum():>8,}  ({df.priced.mean():.2%} of trips)")
print()
print(f"Companies after merging variants: {raw.company.nunique()} -> {df.company_clean.nunique()}")
print(f"Revenue (sum of trip_total): all non-missing ${raw.trip_total.sum():,.0f} | priced basis ${df.loc[df.priced, 'trip_total'].sum():,.0f}")

Trip basis (all trips):                135,768


  excluded: missing money fields            324
  excluded: trip_total == 0                 134
  excluded: trip_total > $500                 8
Priced basis:                          135,302  (99.66% of trips)

Companies after merging variants: 32 -> 30
Revenue (sum of trip_total): all non-missing $4,079,349 | priced basis $4,053,768


# ## 2. When is demand highest?
#
# Trip basis (all 135,768 trips), using the trip start time. "Weekday" = Mon–Fri, "weekend" = Sat–Sun; because there are 5 weekdays and 2 weekend days, the comparisons use *average trips per day*.

In [7]:
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_day = df.groupby("dow").size().reindex(DAYS).rename("trips").to_frame()
by_day["share_%"] = by_day.trips / by_day.trips.sum() * 100
by_day["vs_weekday_avg"] = by_day.trips / by_day.trips.iloc[:5].mean() - 1
print(by_day.to_string(formatters={"trips": "{:,.0f}".format, "share_%": "{:.1f}".format, "vs_weekday_avg": "{:+.0%}".format}))

df["is_weekend"] = df.dow.isin(["Saturday", "Sunday"])
n_days = df.groupby("is_weekend").date.nunique()
avg_hr = (df.groupby(["is_weekend", "hour"]).size().unstack(0) / n_days).rename(columns={False: "weekday_avg", True: "weekend_avg"})
tot_hr = df.groupby("hour").size().rename("all_week_trips")
hourly = avg_hr.join(tot_hr)
hourly["weekday_share_%"] = hourly.weekday_avg / hourly.weekday_avg.sum() * 100
hourly["weekend_share_%"] = hourly.weekend_avg / hourly.weekend_avg.sum() * 100
print()
print("Trips by hour of start (average per day type):")
print(hourly.round(1).to_string())

           trips share_% vs_weekday_avg
dow                                    
Monday    22,190    16.3            +1%
Tuesday   22,040    16.2            +0%
Wednesday 22,694    16.7            +3%
Thursday  22,531    16.6            +3%
Friday    20,226    14.9            -8%
Saturday  12,719     9.4           -42%
Sunday    13,368     9.8           -39%



Trips by hour of start (average per day type):
      weekday_avg  weekend_avg  all_week_trips  weekday_share_%  weekend_share_%
hour                                                                            
0          334.60       366.50            2406             1.50             2.80
1          168.60       290.00            1423             0.80             2.20
2           75.40       197.50             772             0.30             1.50
3           75.60       127.00             632             0.30             1.00
4          120.20       133.00             867             0.50             1.00
5          219.80       129.00            1357             1.00             1.00
6          437.40       203.50            2594             2.00             1.60
7          818.00       262.00            4614             3.70             2.00
8        1,163.20       401.50            6619             5.30             3.10
9        1,308.60       538.50            7620             6.

In [8]:
print("Busiest / quietest hours (all week):")
print(tot_hr.sort_values(ascending=False).head(4).to_dict(), "|", tot_hr.sort_values().head(4).to_dict())
print("Weekday avg per day:", round(df[~df.is_weekend].shape[0] / 5), "| weekend avg per day:", round(df[df.is_weekend].shape[0] / 2))
dh = df.groupby(["date", "hour"]).size().unstack(fill_value=0)
dh.index = dh.index.day_name().str[:3]
print("\nTrips by day x hour (rows Mon..Sun):")
print(dh.to_string())

Busiest / quietest hours (all week):
{17: 9491, 15: 9463, 14: 9268, 16: 9209} | {3: 632, 2: 772, 4: 867, 5: 1357}
Weekday avg per day: 21936 | weekend avg per day: 13044

Trips by day x hour (rows Mon..Sun):
hour   0    1    2    3    4    5    6    7     8     9     10    11    12    13    14    15    16    17    18    19    20   21   22   23
date                                                                                                                                     
Mon   371  179   76   86  145  260  464  771  1168  1339  1392  1349  1455  1469  1571  1642  1619  1597  1409  1144   978  738  547  421
Tue   309  168   63   46   93  167  364  859  1392  1539  1353  1314  1391  1476  1417  1470  1404  1592  1519  1299  1057  749  618  381
Wed   257  114   51   45   94  204  427  889  1234  1407  1307  1363  1474  1528  1563  1523  1597  1615  1531  1334  1131  964  627  415
Thu   325  166   61   88  130  220  490  834  1105  1194  1213  1356  1424  1429  1611  1583  1482  15